# Weather-forecasting model results

Compare the persistence baseline and all four RNN forecasters across the three requested horizons.

## Run training once (only if needed)

Run the next cell only if `outputs/results.csv` is missing or you want fresh results. If training has already run, skip it: the remaining cells load the saved real artifacts.

In [ ]:
# Run only when saved outputs are missing or need regeneration.
!cd .. && python scripts/train.py && python scripts/build_report_assets.py

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
OUTPUTS = Path('../outputs')
if not (OUTPUTS / 'results.csv').exists():
    raise FileNotFoundError('Run the optional training cell first; outputs/results.csv is missing.')
results = pd.read_csv(OUTPUTS / 'results.csv')
results.round(3)

In [ ]:
labels = {'persistence':'Persistence', 'lstm':'LSTM', 'gru':'GRU', 'seq2seq_lstm':'Seq2Seq LSTM', 'attention_seq2seq':'Attention Seq2Seq'}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for axis, target in zip(axes, ['T (degC)', 'rh (%)']):
    for model, label in labels.items():
        subset = results.query('model == @model and target == @target').sort_values('horizon_hours')
        axis.plot(subset.horizon_hours, subset.mae, marker='o', label=label)
    axis.set(title=f'{target}: MAE by horizon', xlabel='Hours ahead', ylabel='MAE', xticks=[6,24,72])
    axis.grid(alpha=.3)
axes[-1].legend(fontsize=8); fig.tight_layout()

In [ ]:
attention = results[results.model.eq('attention_seq2seq')].set_index(['horizon_hours','target'])
plain = results[results.model.eq('seq2seq_lstm')].set_index(['horizon_hours','target'])
ablation = pd.DataFrame({'seq2seq_mae': plain.mae, 'attention_mae': attention.mae})
ablation['relative_change_pct'] = 100 * (ablation.seq2seq_mae - ablation.attention_mae) / ablation.seq2seq_mae
ablation.round(3)

In [ ]:
history = pd.read_csv(OUTPUTS / 'history.csv')
fig, axis = plt.subplots(figsize=(8, 4))
for model, group in history.groupby('model'):
    axis.plot(group.epoch, group.validation_mse, marker='o', label=model.replace('_', ' '))
axis.set(title='Validation learning curves', xlabel='Epoch', ylabel='MSE'); axis.legend(); axis.grid(alpha=.3); fig.tight_layout()